In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dogs-vs-cats-redux-kernels-edition/sample_submission.csv
/kaggle/input/dogs-vs-cats-redux-kernels-edition/train.zip
/kaggle/input/dogs-vs-cats-redux-kernels-edition/test.zip


In [2]:
#IMport the os module to interact with the operating system
# Used for file path operations, directory creation and file system navigation
import os

#Imports numpy for numerical operations and array manipulations
#Essential for mathematical computations and data preprocessing
import numpy as np

#Imports pandas for data manipulation and analysis
#Used for creating dataframes, reading csv files and data organizatino
import pandas as pd

#Imports matplotlib for data visualization and plotting
#Used to create charts, graphs, and visualize training metrics
import matplotlib.pyplot as plt

##Import PIL(Python imaging library) for image preprocessing
#Used to load, manipulate, and save image files
from PIL import Image

#Imports tqdm for progress bars during loops
#Provides visual feedback during long-running operations training
from tqdm import tqdm

#Imports zipfile to handle compressed ZIOP archives
#Used to copying, moving and removing files and directories
import zipfile

#Imports shutil for high level file operations
#Used for copying, moving and removing files and directories
import shutil

#Imports pytorch-the main deep learning framework
#Core library for building adn training neural networks
import torch

#Imports neural network module from pytorch
#Contains building blocks like layers, loss functions and activations
import torch.nn as nn

#Imports optimization algorithms from pytorch
#Provides optimizers like Adam, SGD, for training neural networks
import torch.optim as optim

#Imports Dataset and Dataloader classes from Pytorch
#Dataset: Base class for creating custom datasets
#Dataloader: Efficiently loads data in batches during training.
from torch.utils.data import Dataset, DataLoader

# torchvision- Pytoch's computer vision library
#Provides pre-tarined models, datasets and image transformations
import torchvision

#Imports transforms and models from torhcvision
#transforms : Image preprocessing operatinos(resize, normalize, augment)
#models: Pre-trained neural network architectures(ResNet, VGG, etc.)
from torchvision import transforms, models

print("PyTorch Version:", torch.__version__)

#Checks if cuda is available
print("CUDA Available:", torch.cuda.is_available())

#All tensors and models will be moved to this device available for computation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

PyTorch Version: 2.6.0+cu124
CUDA Available: False
Device: cpu


In [ ]:
import random  # Added for augmentation

# ==================== AUGMENTATION HELPER FUNCTIONS ====================
# These functions create harder training examples by mixing images

def mixup_data(x, y, alpha=1.0):
    """
    MixUp: Blends two images together
    Example: 60% cat + 40% dog = mixed training image
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Calculate loss for mixed images"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def cutmix_data(x, y, alpha=1.0):
    """
    CutMix: Cuts a patch from one image and pastes onto another
    Example: Replace part of cat image with part of dog image
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    # Calculate box dimensions
    W, H = x.size()[2], x.size()[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    
    # Random box position
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    # Apply cutmix
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    
    # Adjust lambda based on actual box size
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

print("✓ Augmentation functions loaded successfully!")
print("  - MixUp: Blends images together")
print("  - CutMix: Cuts and pastes image patches")


In [3]:
# Paths for Redux competition
KAGGLE_INPUT = '/kaggle/input/dogs-vs-cats-redux-kernels-edition'
WORK_DIR = '/kaggle/working'
TRAIN_DIR = f'{WORK_DIR}/train'
TEST_DIR = f'{WORK_DIR}/test'

# Extract training data
print("\nExtracting train.zip...")

#Checks if training directory doesn't already exist to avoid duplicate extraction
if not os.path.exists(TRAIN_DIR):
    #Open the train.zip file in read mode using zipfile module
    with zipfile.ZipFile(f'{KAGGLE_INPUT}/train.zip', 'r') as zip_ref:
        #Extract all contents of the zip file in the working directory
        zip_ref.extractall(WORK_DIR)
    print("Training data extracted")
else:
    print("Training directory already exists")

# Extract test data
print("Extracting test.zip...")
#Checks if Test directory doesn't already exists to avoid duplicate extraction.
if not os.path.exists(TEST_DIR):
    #Open the test.zip file in read mode using zipfile module
    with zipfile.ZipFile(f'{KAGGLE_INPUT}/test.zip', 'r') as zip_ref:
        #Extracts all contents of the zip file in the working directory
        zip_ref.extractall(WORK_DIR)
    print("Test data extracted")
else:
    print("Test directory already exists")

# Organize training data into cat/dog folders
print("\nOrganizing training data...")

#Defines the directory path where the cat images will be stored
cat_dir = f'{TRAIN_DIR}/cats'
#Defines the direcory path where the dog images will be stored
dog_dir = f'{TRAIN_DIR}/dogs'

#creates the cats directory if doesn't exist . exist_ok=True prevents error if it doesn't exist
os.makedirs(cat_dir, exist_ok=True)

#create the dogs directory if it doesn't exist. exist_ok=True prevents error if it doesn't exist
os.makedirs(dog_dir, exist_ok=True)

#Get a list of all JPG image files in the training directory, excluding subdirectories
train_files = [f for f in os.listdir(TRAIN_DIR) if f.endswith('.jpg')]

#CHecks if there are any images that need to be organized
if len(train_files) > 0:
    #Display how many images were found that need organizing
    print(f"Found {len(train_files)} images to organize")

    #Loop through each image file with a progress bar( tqdm shows completion status)
    for filename in tqdm(train_files, desc="Organizing"):
        #Create the full source path of the current image file
        src = os.path.join(TRAIN_DIR, filename)

        #Checks if the filename contains cat to identify cat images
        if 'cat' in filename:
            #Set the destination path to the cats directory
            dst = os.path.join(cat_dir, filename)
        else:
            #Otherwise the destination path to the dogs directory
            dst = os.path.join(dog_dir, filename)

        #Verify the source file exists before attempting to move it
        if os.path.exists(src):
            # This ACTUALLY moves the physical image file
            # FROM location "src" TO location "dst"
            shutil.move(src, dst)
else:
    #Inform user that images are already organized  in their respective folders
    print("Images already organized")

#Counts the number of cat images by checking files in the cats directory
num_cats = len(os.listdir(cat_dir))
num_dogs = len(os.listdir(dog_dir))

print(f"\nCats: {num_cats} images")
print(f"Dogs: {num_dogs} images")
print(f"Total: {num_cats + num_dogs} images")



Extracting train.zip...
Training data extracted
Extracting test.zip...
Test data extracted

Organizing training data...
Found 25000 images to organize


Organizing: 100%|██████████| 25000/25000 [00:00<00:00, 28791.76it/s]


Cats: 12500 images
Dogs: 12500 images
Total: 25000 images


In [4]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
NUM_WORKERS = 2

print(f"\nConfiguration:")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")


Configuration:
Image Size: 224x224
Batch Size: 32
Epochs: 30
Learning Rate: 0.001


In [5]:
# ==================== DATA TRANSFORMS ====================
# UPDATED: Advanced augmentation for better model learning

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.8, 1.2),
        shear=10
    ),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.1
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.33), ratio=(0.3, 3.3))
])

# Validation transforms stay the same
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✓ Transforms configured")
print("  - Training: Advanced augmentation enabled")
print("  - Validation: Standard transforms")


Data transforms created


In [6]:
# Custom Dataset class
class CatsDogsDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        
        cat_dir = os.path.join(root_dir, 'cats')
        if os.path.exists(cat_dir):
            for img_name in os.listdir(cat_dir):
                if img_name.endswith('.jpg'):
                    self.images.append(os.path.join(cat_dir, img_name))
                    self.labels.append(0)
        
        dog_dir = os.path.join(root_dir, 'dogs')
        if os.path.exists(dog_dir):
            for img_name in os.listdir(dog_dir):
                if img_name.endswith('.jpg'):
                    self.images.append(os.path.join(dog_dir, img_name))
                    self.labels.append(1)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [7]:
print("Dataset class created")

# Create datasets
print("\nCreating datasets...")
full_dataset = CatsDogsDataset(TRAIN_DIR, transform=train_transforms)
print(f"Total images loaded: {len(full_dataset)}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Dataset class created

Creating datasets...
Total images loaded: 25000
Train samples: 20000
Validation samples: 5000


In [8]:
# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("DataLoaders created")


DataLoaders created


In [11]:
# Build model - Fixed for Kaggle internet restrictions
print("\nBuilding ResNet50 model...")

# Try with pretrained weights if internet is enabled
try:
    from torchvision.models import ResNet50_Weights
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    print("Loaded pretrained weights")
    use_pretrained = True
except:
    print("Could not download pretrained weights, using random initialization")
    model = models.resnet50(weights=None)
    use_pretrained = False

# Only freeze if we have pretrained weights
if use_pretrained:
    for param in model.parameters():
        param.requires_grad = False
    print("Froze base layers (using transfer learning)")
else:
    # Training from scratch - keep all layers trainable
    for param in model.parameters():
        param.requires_grad = True
    print("Training all layers from scratch")

# Replace final layer
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Dropout(0.5),
    nn.Linear(256, 1),
    nn.Sigmoid()
)

model = model.to(device)

print("Model created and moved to device")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.BCELoss()

# Use lower learning rate if training from scratch
lr = 0.0001 if not use_pretrained else 0.001
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max', 
    factor=0.5, 
    patience=5
)

print(f"Learning rate: {lr}")
print("Loss, optimizer, and scheduler ready")


Building ResNet50 model...


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


Could not download pretrained weights, using random initialization
Training all layers from scratch
Model created and moved to device
Total parameters: 24,690,241
Trainable parameters: 24,690,241
Learning rate: 0.0001
Loss, optimizer, and scheduler ready


In [12]:
# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        predictions = (outputs > 0.5).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{correct/total:.4f}'
        })
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [13]:
# Validation function
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            predictions = (outputs > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{correct/total:.4f}'
            })
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


In [14]:
# ==================== VALIDATION FUNCTION ====================
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            predictions = (outputs > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
            pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [15]:
print("Training functions defined")

# Training loop
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
patience_counter = 0
early_stop_patience = 10

Training functions defined

STARTING TRAINING


In [ ]:
for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print('='*60)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_acc)
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} ({val_acc*100:.2f}%)")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"New best model saved! Val Acc: {val_acc:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{early_stop_patience}")
    
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        break



Epoch 1/30


Training:  73%|███████▎  | 458/625 [1:18:10<28:56, 10.40s/it, loss=0.6903, acc=0.5608]

In [ ]:
# ==================== TEST DATASET ====================
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.images = sorted(os.listdir(test_dir), key=lambda x: int(x.split('.')[0]))
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        img_id = int(img_name.split('.')[0])
        return image, img_id


In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

model.load_state_dict(torch.load('best_model.pth'))
print("Best model loaded")

# Plot training history
print("\nPlotting training history...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = range(1, len(history['train_acc']) + 1)

ax1.plot(epochs_range, history['train_acc'], 'o-', label='Train Accuracy', linewidth=2)
ax1.plot(epochs_range, history['val_acc'], 's-', label='Val Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', linewidth=2)
ax2.plot(epochs_range, history['val_loss'], 's-', label='Val Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history plot saved")


In [ ]:
# Test dataset class
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.images = sorted(
            [f for f in os.listdir(test_dir) if f.endswith('.jpg')],
            key=lambda x: int(x.split('.')[0])
        )
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        img_id = int(img_name.split('.')[0])
        return image, img_id

print("Test dataset class created")

In [ ]:
# Generate predictions
print("\n" + "="*60)
print("GENERATING PREDICTIONS")
print("="*60)

test_dataset = TestDataset(TEST_DIR, transform=val_transforms)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(f"Test samples: {len(test_dataset)}")

model.eval()
predictions = []
image_ids = []

print("\nProcessing test images...")
with torch.no_grad():
    for images, ids in tqdm(test_loader):
        images = images.to(device)
        outputs = model(images)
        
        predictions.extend(outputs.cpu().numpy().flatten().tolist())
        image_ids.extend(ids.numpy().tolist())

print(f"Generated {len(predictions)} predictions")

In [ ]:
# Create submission file
print("\nCreating submission file...")

submission = pd.DataFrame({
    'id': image_ids,
    'label': predictions
})

submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("SUBMISSION FILE CREATED")
print("="*60)
print(f"\nFile: submission.csv")
print(f"Total predictions: {len(submission)}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))
print(f"\nLast 10 predictions:")
print(submission.tail(10))
print(f"\nPrediction Statistics:")
print(f"Min:  {submission['label'].min():.6f}")
print(f"Max:  {submission['label'].max():.6f}")
print(f"Mean: {submission['label'].mean():.6f}")
print(f"Std:  {submission['label'].std():.6f}")

dog_count = (submission['label'] > 0.5).sum()
cat_count = (submission['label'] <= 0.5).sum()
print(f"\nPredicted Distribution:")
print(f"Dogs: {dog_count} ({dog_count/len(submission)*100:.1f}%)")
print(f"Cats: {cat_count} ({cat_count/len(submission)*100:.1f}%)")

print("\n" + "="*60)
print("COMPLETE")
print("="*60)
print("\nNext Steps:")
print("1. Download submission.csv from Output section")
print("2. Submit to Kaggle competition")
print("="*60)